# Baomai: repeat the box-label comparison at seeds 43 and 44

**Select a GPU runtime, choose Run all, then return one ZIP.** This notebook runs the two additional, predeclared seed pairs for `box-supervision-repeatability-v1`. The completed seed-42 experiment remains historical evidence; it is not rerun here.

The fixed sequence is a four-batch candidate resource trial at seed 43, then seed-43 reference/candidate and seed-44 reference/candidate. Every full run starts freshly from epoch173 and completes the unchanged 30-epoch low-bias schedule. The only planned difference between seed pairs is their declared seed. Within each pair, the only data difference is the same 16 reviewed box-edge corrections across three training label files.

The notebook packages **four complete final-epoch models with their raw provenance, logs and results** into `baomai-box-repeatability-results.zip`. Both pairs must pass verification before a ZIP is created. It does not evaluate accuracy, select the best seed, change annotations or change the working model.

Keep the same GPU session, software, batch size, image size, schedule and candidate throughout. A runtime/resource failure is not evidence about model quality. Preserve its receipt and log.


## 1. Check the GPU and create a fresh workspace

In Colab, select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Leave the session unchanged through all four runs and packaging.

This notebook refuses to reuse its workspace automatically. After an interrupted run, use the manual retry instructions below to preserve completed runs in the same unchanged session. Start a new whole workspace if the session or software changed.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import subprocess
import sys
import time

SOURCE_REVISION = "SOURCE_REVISION_PLACEHOLDER"
STUDY_ID = "box-supervision-repeatability-v1"
RECIPE_BY_SEED = {43: "box-supervision-warmup-seed43-v1", 44: "box-supervision-warmup-seed44-v1"}
CANDIDATE_SHA256 = "b61d012052001377089ad048ba13da14b8cd19d665ac8f0ef7828e45c530ec3e"
CANDIDATE_MANIFEST = "data/candidates/box-refinement-2026-11-07/manifest.json"
RUN_BASE = Path("/content/baomai-box-repeatability-20261107")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the handoff notebook with its full source commit pinned."
assert sys.version_info >= (3, 11), "Python 3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run([sys.executable, "-c", "import torch, torchvision; "
     "assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'; "
     "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda); "
     "print('GPU:', torch.cuda.get_device_name(0)); "
     "print('CUDA calculation:', (torch.ones(1, device='cuda:0') * 2).item())"])
run(["nvidia-smi"])
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    raise FileExistsError(f"Workspace already exists: {RUN_BASE}. Preserve its results and choose a new RUN_BASE.")
RUN_BASE.mkdir(parents=True, exist_ok=False)
REPO = RUN_BASE / "dl"
print("Workspace:", RUN_BASE)


## 2. Download the pinned source and verify all original inputs

Git supplies the immutable code, original labels and unchanged reviewed overlay. Selective Git LFS obtains the 157 original PNGs and epoch173 initialization. Each repeatability recipe is checked against the original box recipe with exactly its declared seed substitution. The candidate manifest must have the predeclared byte hash.


In [ ]:
lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
run(["git", "init", REPO], env=git_env)
run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
run(["git", "fetch", "--depth=1", "origin", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"],
    cwd=REPO, text=True).splitlines())
assert len(reference_manifest["records"]) == len(expected_images) == 157
assert tracked_images == expected_images, "The LFS image selection differs from the frozen manifest."
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt",
     "--exclude="], cwd=REPO)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REVISION
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()

input_preflight = """
import json
from pathlib import Path
import sys
sys.dont_write_bytecode = True
root = Path.cwd()
sys.path.insert(0, str(root / 'scripts'))
from evaluate_pose import preflight_dataset
from stage_pose_training import source_state
from pose_checkpoint import BOX_RECIPE_ID, BOX_REPEATABILITY_RECIPES, get_recipe, recipe_identity
from evaluate_pose import file_sha256
from package_box_repeatability import CANDIDATE_SHA256, STUDY_ID
candidate = root / sys.argv[1]
expected_recipes = {43: "box-supervision-warmup-seed43-v1", 44: "box-supervision-warmup-seed44-v1"}
assert BOX_REPEATABILITY_RECIPES == expected_recipes
assert STUDY_ID == "box-supervision-repeatability-v1"
assert file_sha256(candidate) == CANDIDATE_SHA256 == sys.argv[2]
proposal = json.loads(candidate.read_text())
assert proposal['kind'] == 'training_box_overlay'
assert proposal['candidate_id'] == 'box-refinement-2026-11-07'
edge_counts = {change['image_id']: sum(len(row['box_edges_changed']) for row in change['rows'])
               for change in proposal['changes']}
assert edge_counts == {'new_000030': 8, 'new_000012': 6, 'new_000128': 2}, edge_counts
state = source_state(root, candidate)
initial = preflight_dataset(root, require_weights=True)
assert len(state['records']) == 157
assert state['counts']['reference'] == state['counts']['candidate']
assert state['counts']['reference']['train']['images'] == 137
assert state['counts']['reference']['val']['images'] == 20
assert state['source']['initial_checkpoint_sha256'] == initial['model_sha256']
assert Path(initial['weights']) == root / 'weights/cassette_lower_epoch173.pt'
for seed, recipe_id in expected_recipes.items():
    assert get_recipe(recipe_id) == {**get_recipe(BOX_RECIPE_ID), 'seed': seed}
print(json.dumps({'status': 'verified', 'original_images': len(state['records']),
                  'box_edge_corrections': edge_counts, 'label_counts': state['counts'],
                  'initial_checkpoint_sha256': initial['model_sha256'],
                  'weights_verified': initial['weights_verified'],
                  'recipes': {seed: recipe_identity(recipe_id) for seed, recipe_id in expected_recipes.items()}}, indent=2))
"""
run([sys.executable, "-c", input_preflight, CANDIDATE_MANIFEST, CANDIDATE_SHA256], cwd=REPO)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified clean source:", actual_revision)


## 3. Preserve CUDA PyTorch and install the pinned trainer

The installed `torch` and `torchvision` versions are constrained before installing `ultralytics==8.4.182`. Do not change this environment between runs. Resolve a dependency conflict before training instead of silently replacing the CUDA stack.

The host-RAM and time settings are stop budgets, not promised usage or duration. A short trial cannot certify a full run's validation/finalization memory requirements.


In [ ]:
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text(
    ''.join(f'{name}=={version(name)}\n' for name in ('torch', 'torchvision')), encoding='utf-8'
)
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; assert torch.cuda.is_available(); "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__, torch.cuda.get_device_name(0))"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or select a larger-memory runtime, then rerun this cell."
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Time budgets: 600 seconds for the trial; 21600 seconds per complete run (four runs).")


## 4. Run the real four-batch resource trial

The seed-43 candidate trial uses the full 30-epoch schedule but stops after four real batches. It must record optimizer updates, changed parameters, the selected seed/recipe and verified inputs. Its weights are never used to initialize a full run, and it cannot produce an eligible completed checkpoint.

Progress is printed every 20 seconds. On failure the cell displays the receipt error and tail of `train.log`, then stops. No automatic retry, seed substitution, checkpoint resume or output overwrite is performed.


In [ ]:
def train_one(folder, arm, *, seed, trial=False):
    assert arm in ("reference", "candidate")
    assert seed in (43, 44), "Use only the two declared repeatability seeds."
    recipe_id = RECIPE_BY_SEED[seed]
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        raise FileExistsError(f"Preserve {output}; retry only the failed cell with a new folder name.")
    command = [sys.executable, "scripts/train_pose_comparison.py",
               "--trial" if trial else "--train", "--arm", arm, "--recipe", recipe_id,
               "--candidate", CANDIDATE_MANIFEST,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, arm, "seed", seed, recipe_id, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}

    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass

    started = time.monotonic()
    last_notice = -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; status={progress.get('status')}; "
                          f"completed epochs={len(progress.get('completed_epochs', []))}/30; "
                          f"batches={progress.get('batches_completed', 0)}; "
                          f"updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: {elapsed:.0f}s; verifying/staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            receipt_file = output / "training-receipt.json"
            if receipt_file.exists():
                failed_receipt = json.loads(receipt_file.read_text())
                for key in ("error", "input_verification_error", "checkpoint_manifest_error"):
                    if failed_receipt.get(key):
                        print(f"{key}: {failed_receipt[key]}", flush=True)
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed. Preserve its receipt/log and use a new output directory for a retry.")
    except BaseException:
        # Stop this invocation's descendants if the notebook cell is interrupted.
        # Partial runs remain ineligible; no receipt is fabricated here.
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = json.loads((output / "training-receipt.json").read_text())
    assert receipt["status"] == ("trial-complete" if trial else "complete")
    assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
    assert receipt["arm"] == arm
    assert receipt["recipe_id"] == recipe_id
    assert receipt["initial_checkpoint_sha256"] == reference_manifest["model_sha256"]
    assert receipt["source"]["revision"] == SOURCE_REVISION
    assert receipt["source"]["working_tree_status"] == ""
    assert receipt["training_settings"]["seed"] == seed
    assert receipt["training_settings"]["epochs"] == 30
    assert receipt["training_settings"]["warmup_bias_lr"] == 0.0001
    if trial:
        assert seed == 43 and arm == "candidate"
        assert receipt["batches_completed"] == 4 and receipt["completed_epochs"] == []
        assert receipt["checkpoint_selection"] == "none"
    else:
        assert receipt["completed_epochs"] == list(range(1, 31))
    assert (output / "checkpoint-manifest.json").exists() is (not trial)
    return output

TRIAL = train_one("resource-trial", "candidate", seed=43, trial=True)
print("Resource trial passed. This is not a completed model.")


## 5. Complete the two matched seed pairs

The next four cells run sequentially: seed 43 reference, seed 43 candidate, seed 44 reference, seed 44 candidate. Each starts freshly from the same epoch173 bytes. Neither arm starts from a trial, another arm or a previous seed. Selection remains the finalized epoch-30 `last.pt`.

Separate cells preserve a simple recovery path: a failed cell gets a new output folder; already completed siblings are retained and independently reverified by the final packager.


In [ ]:
SEED43_REFERENCE = train_one("seed-43/reference", "reference", seed=43)


In [ ]:
SEED43_CANDIDATE = train_one("seed-43/candidate", "candidate", seed=43)


In [ ]:
SEED44_REFERENCE = train_one("seed-44/reference", "reference", seed=44)


In [ ]:
SEED44_CANDIDATE = train_one("seed-44/candidate", "candidate", seed=44)


### If one full run fails

Keep its existing folder and log. Do not rerun completed training cells or turn on checkpoint resume. In the **same unchanged session**, inspect the failure, then change only the failed cell's folder argument to a new name such as `seed-43/candidate-retry-001`. Keep its seed, arm and recipe unchanged, run that cell again from epoch173, and then run any still-missing later cells. The successful assignment updates that arm's path used by packaging.

For example, after a failed seed-43 candidate cell, its explicit retry is:

```python
SEED43_CANDIDATE = train_one("seed-43/candidate-retry-001", "candidate", seed=43)
```

The packager rechecks every retained complete run, all four exact recipes, the original candidate hash, source/runtime identity and settings. It refuses mismatches or missing/partial arms. If software/GPU session changes, preserve the old evidence and start a fresh whole study workspace; do not combine environments. The notebook never chooses among several successful runs by their metrics.


## 6. Verify all four runs and create one ZIP

The study packer requires exactly the declared seed-43 and seed-44 pairs. It retains the existing within-pair checks and compares both pairs' source, runtime, dataset proposal and resolved settings, allowing only the declared seed and run paths to differ. Legacy seed 42 cannot stand in for a new run.

The flat ZIP contains `seed-43/reference`, `seed-43/candidate`, `seed-44/reference` and `seed-44/candidate`, each with final weights, checkpoint/receipt/staging manifests, epoch results and the raw training log. The four-batch trial receipt, staging manifest and log are supplemental, explicitly ineligible as a completed checkpoint. Source images and initialization weights are omitted.

An existing ZIP is never overwritten. After successful packaging, rerun only the download cell if the browser transfer needs another attempt.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-box-repeatability-results.zip"
run([sys.executable, "scripts/package_box_repeatability.py",
     "--seed-43", SEED43_REFERENCE, SEED43_CANDIDATE,
     "--seed-44", SEED44_REFERENCE, SEED44_CANDIDATE,
     "--trial", TRIAL, "--output", RESULTS_ZIP], cwd=REPO)
with RESULTS_ZIP.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()
print("Study:", STUDY_ID)
print("Return:", RESULTS_ZIP.name)
print("Bytes:", RESULTS_ZIP.stat().st_size)
print("SHA-256:", digest)


## 7. Download and return the ZIP

Attach **`baomai-box-repeatability-results.zip`** in chat or return it in a new result PR before closing the runtime. Keep the original ZIP unchanged. If the browser blocks the transfer, rerun this download cell; it does not train or repackage anything.


In [ ]:
assert RESULTS_ZIP.is_file(), "Finish the packaging cell first."
from google.colab import files
files.download(str(RESULTS_ZIP))


### What happens after the return

Evaluate each new candidate against its same-seed original-label control under the unchanged validation images, scorer and thresholds, and retain the completed seed-42 result as a separate historical pair. Summarize the predefined paired effects for both prospective seeds 43 and 44; show seed 42 separately as historical context. Do not select a best seed or change tolerances after inspecting the outcomes.

This study addresses sensitivity to training randomness on an already-used development split. It cannot establish new-scene accuracy, physical subpixel significance or deployment readiness. The current working model remains epoch173 until a separately justified decision changes it.
